# Statistics

This notebook covers statistics on trees encoded as Phylo2Vec vectors (topology only) or matrices (with branch lengths):
* Distances between leaves (cophenetic distances)
* Phylogenetic covariance and precision matrices
* Incidence matrices
* Tree balance indices
* Distances between trees (Robinson-Foulds)

To get familiar with the Phylo2Vec format first, see [Getting started](getting-started.ipynb).

## Imports

In [ ]:
import numpy as np

from ete4 import Tree

import phylo2vec as p2v


def plot_tree(newick):
    print(Tree(newick))

We use the following tree with 7 leaves, and a tree with 5 leaves and branch lengths:

In [ ]:
v_fixed = np.array([0, 2, 2, 5, 4, 1])

plot_tree(p2v.to_newick(v_fixed))

   ╭─┬╴0
 ╭─┤ ╰─┬╴1
 │ │   ╰╴6
─┤ ╰─┬╴4
 │   ╰╴5
 ╰─┬╴2
   ╰╴3


In [ ]:
m5 = np.array(
    [
        [0.0, 0.05586577, 0.73334581],
        [0.0, 0.38201654, 0.61142737],
        [4.0, 0.85919964, 0.30591026],
        [0.0, 0.25750902, 0.02408398],
    ]
)

print(p2v.to_newick(m5))

((((0:0.05586577,4:0.73334581)5:0.38201654,2:0.61142737)6:0.85919964,1:0.30591026)7:0.25750902,3:0.02408398)8;


## 1. Cophenetic distances

The [cophenetic distance](https://en.wikipedia.org/wiki/Cophenetic) between two leaves is the length of the path between them. With a vector, all branch lengths are 1.

In [ ]:
from phylo2vec.stats import pairwise_distances

print(f"Cophenetic distance matrix (topology):\n{pairwise_distances(v_fixed, metric='cophenetic')}")

Cophenetic distance matrix (topology):
[[0. 3. 5. 5. 4. 4. 3.]
 [3. 0. 6. 6. 5. 5. 2.]
 [5. 6. 0. 2. 5. 5. 6.]
 [5. 6. 2. 0. 5. 5. 6.]
 [4. 5. 5. 5. 0. 2. 5.]
 [4. 5. 5. 5. 2. 0. 5.]
 [3. 2. 6. 6. 5. 5. 0.]]


In [ ]:
print(
    f"Cophenetic distance matrix (rounded):\n{pairwise_distances(m5, metric='cophenetic').round(3)}"
)

Cophenetic distance matrix (rounded):
[[0.    1.603 1.049 1.579 0.789]
 [1.603 0.    1.777 0.588 2.28 ]
 [1.049 1.777 0.    1.752 1.727]
 [1.579 0.588 1.752 0.    2.256]
 [0.789 2.28  1.727 2.256 0.   ]]


## 2. Covariance and precision matrices

Use `cov` to compute the phylogenetic variance-covariance matrix (similar to `vcv` in the R package [ape](https://cran.r-project.org/package=ape)). Entry $(i, j)$ is the length of the path from the root shared by leaves $i$ and $j$, so the diagonal holds the distance from the root to each leaf.

In [ ]:
from phylo2vec.stats import cov

c = cov(v_fixed)

print(c)

[[3. 2. 0. 0. 1. 1. 2.]
 [2. 4. 0. 0. 1. 1. 3.]
 [0. 0. 2. 1. 0. 0. 0.]
 [0. 0. 1. 2. 0. 0. 0.]
 [1. 1. 0. 0. 3. 2. 1.]
 [1. 1. 0. 0. 2. 3. 1.]
 [2. 3. 0. 0. 1. 1. 4.]]


Use `precision` to compute the precision matrix, i.e., the inverse of the covariance matrix:

In [ ]:
from phylo2vec.stats import precision

p = precision(v_fixed)

print(p.round(3))

assert np.allclose(p @ c, np.eye(len(c)))  # p is the inverse of c

[[ 0.564 -0.145  0.     0.    -0.055 -0.055 -0.145]
 [-0.145  0.618  0.     0.    -0.018 -0.018 -0.382]
 [ 0.     0.     0.667 -0.333  0.     0.     0.   ]
 [ 0.     0.    -0.333  0.667  0.     0.     0.   ]
 [-0.055 -0.018  0.     0.     0.618 -0.382 -0.018]
 [-0.055 -0.018  0.     0.    -0.382  0.618 -0.018]
 [-0.145 -0.382  0.     0.    -0.018 -0.018  0.618]]


Both also accept matrices:

In [ ]:
print(cov(m5).round(3))

[[1.555 0.258 1.117 0.    1.499]
 [0.258 0.563 0.258 0.    0.258]
 [1.117 0.258 1.728 0.    1.117]
 [0.    0.    0.    0.024 0.   ]
 [1.499 0.258 1.117 0.    2.232]]


## 3. Incidence matrix

Use `incidence` to compute the incidence matrix of a tree topology. Rows are nodes, and column $j$ is the branch leading to node $j$: it has a 1 at node $j$ and a -1 at its parent. For a tree with $n$ leaves, the matrix has shape $(2n-1, 2n-2)$.

In [ ]:
from phylo2vec.stats import incidence

v3 = np.array([0, 1])

print(p2v.to_newick(v3))
print(incidence(v3, format="dense"))

(0,(1,2)3)4;
[[ 1  0  0  0]
 [ 0  1  0  0]
 [ 0  0  1  0]
 [ 0 -1 -1  1]
 [-1  0  0 -1]]


For large trees, use the sparse formats: `coo` (default), `csr` or `csc`. For instance, with [SciPy](https://scipy.org/):

In [ ]:
from scipy.sparse import coo_array

data, rows, cols = incidence(v_fixed, format="coo")

inc = coo_array((data, (rows, cols)))

assert np.array_equal(inc.toarray(), incidence(v_fixed, format="dense"))

## 4. Tree balance

Balance indices summarise how symmetric a tree is:
* the Sackin index (`sackin`) is the sum of the leaf depths: the higher, the less balanced
* the B2 index (`b2`) is $\sum_i d_i 2^{-d_i}$, where $d_i$ is the depth of leaf $i$: the higher, the more balanced
* the leaf depth variance (`leaf_depth_variance`) is 0 for perfectly balanced trees

These indices only depend on the topology, so matrices are also accepted. Let us compare a ladder (caterpillar) tree with a perfectly balanced tree:

In [ ]:
from phylo2vec.stats import b2, leaf_depth_variance, sackin

ladder = np.array([0, 1, 2, 3, 4, 5, 6])
balanced = np.array([0, 2, 2, 6, 4, 6, 6])

for name, tree in [("Ladder", ladder), ("Balanced", balanced)]:
    print(f"{name} tree:")
    plot_tree(p2v.to_newick(tree))
    print(
        f"Sackin: {sackin(tree)}, B2: {b2(tree):.3f}, "
        f"leaf depth variance: {leaf_depth_variance(tree):.3f}\n"
    )

Ladder tree:
─┬╴0
 ╰─┬╴1
   ╰─┬╴2
     ╰─┬╴3
       ╰─┬╴4
         ╰─┬╴5
           ╰─┬╴6
             ╰╴7
Sackin: 35, B2: 1.984, leaf depth variance: 4.484

Balanced tree:
   ╭─┬╴0
 ╭─┤ ╰╴1
 │ ╰─┬╴2
─┤   ╰╴3
 │ ╭─┬╴4
 ╰─┤ ╰╴5
   ╰─┬╴6
     ╰╴7
Sackin: 24, B2: 3.000, leaf depth variance: 0.000



## 5. Comparing trees: Robinson-Foulds distance

Another important tree distance metric is the [Robinson-Foulds (RF) distance](https://en.wikipedia.org/wiki/Robinson%E2%80%93Foulds_metric), which counts the number of bipartitions (splits) that differ between two tree topologies.

Use `robinson_foulds` to compute the RF distance between two trees.

In [ ]:
from phylo2vec.stats import robinson_foulds

# Sample two random trees with 10 leaves
v1 = p2v.sample_vector(10)
v2 = p2v.sample_vector(10)

print(f"Tree 1: {repr(v1)}")
print(f"Tree 2: {repr(v2)}")

# Compute RF distance
rf_dist = robinson_foulds(v1, v2)
print(f"Robinson-Foulds distance: {rf_dist}")

# Normalized RF distance (range [0, 1])
rf_norm = robinson_foulds(v1, v2, normalize=True)
print(f"Normalized RF distance: {rf_norm:.3f}")

# RF distance of a tree with itself is always 0
rf_same = robinson_foulds(v1, v1)
print(f"RF distance (same tree): {rf_same}")

Tree 1: array([0, 1, 2, 2, 7, 3, 7, 4, 1])
Tree 2: array([0, 0, 2, 3, 5, 3, 4, 3, 0])
Robinson-Foulds distance: 14.0
Normalized RF distance: 1.000
RF distance (same tree): 0.0
